# 📦 Data Contract — Análise Long Neck WSNP (NENO)

**Projeto:** Planejamento de Supply Chain — Embalagens Long Neck — Região NENO  
**Empresa:** Ambev  
**Arquivo de dados:** `Analise_LongNeck_WSNP_-_Sem_repostas.xlsb`  
**Data de referência:** Fevereiro/2026 (W0: 02/02/2026)

---

Este notebook documenta o **Data Contract** do projeto, descrevendo cada variável presente nas abas do arquivo Excel de planejamento logístico da Ambev para a linha Long Neck na região Nordeste (NENO).

## 1. Contexto da Dor do Cliente

A região **Nordeste (NENO)** da Ambev apresenta crescimento consistente no volume de marcas premium, com destaque para a embalagem **Long Neck (LN)**. A capacidade produtiva local está próxima do limite, gerando necessidade de transferências externas (principalmente de SP e MG) para suprir a demanda.

**Situação-problema:** As regiões Geo Nordeste e Geo Norte solicitaram um aumento de **30% na demanda de Malzbier Brahma para Fevereiro/2026**, além de sinalização de **+10% no total de LN a partir de Março**. Isso pressiona ainda mais a capacidade já limitada da região.

**Perguntas a responder:**
1. Devemos seguir com os incentivos comerciais?
2. Qual será o plano de produção e transferência para atendimento das novas demandas?
3. Quanto vai custar a operação?
4. Quais os riscos envolvidos?

**Premissas:**
- Restrições produtivas e logísticas vigentes
- Custos de transferência ex-Nordeste (ex-NE)
- **DOI mínimo de 12 dias** para a região NENO

## 2. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

print('Bibliotecas importadas com sucesso.')

: 

## 3. Data Contract

O Data Contract é a documentação formal de cada variável presente no dataset. Garante alinhamento entre as áreas de negócio e dados, evitando ambiguidades na análise.

### 3.1 Dimensões Gerais

In [ ]:
dimensoes_gerais = pd.DataFrame([
    {'Nome da Coluna': 'GEO/REG',       'Tipo': 'str',   'Descrição': 'Sub-região geográfica de destino (ex: NE Norte, NE Sul, Mapapi, NO Araguaia, NO Centro)'},
    {'Nome da Coluna': 'SKU',           'Tipo': 'str',   'Descrição': 'Código identificador do produto no sistema (Stock Keeping Unit)'},
    {'Nome da Coluna': 'Desc Produto',  'Tipo': 'str',   'Descrição': 'Descrição completa do produto (ex: MALZBIER BRAHMA LN355ML SIX PAC BAND C/4)'},
    {'Nome da Coluna': 'Cod Produto',   'Tipo': 'str',   'Descrição': 'Código numérico do produto no sistema'},
    {'Nome da Coluna': 'Linha',         'Tipo': 'str',   'Descrição': 'Linha de envase associada ao produto (ex: L541)'},
    {'Nome da Coluna': 'Container Type','Tipo': 'str',   'Descrição': 'Tipo de embalagem Long Neck (LONG NECK 330ML, 355ML ou 269ML)'},
])

display(dimensoes_gerais)

### 3.2 Produção

In [ ]:
producao = pd.DataFrame([
    {'Nome da Coluna': 'PROD',                  'Tipo': 'float', 'Descrição': 'Volume total de produção planejada para o período (KHL)'},
    {'Nome da Coluna': 'Real',                  'Tipo': 'float', 'Descrição': 'Volume efetivamente realizado de produção no período (KHL)'},
    {'Nome da Coluna': 'WSNP',                  'Tipo': 'float', 'Descrição': 'Weekly Supply Network Planning — volume programado para produção na semana corrente (KHL)'},
    {'Nome da Coluna': '1W',                    'Tipo': 'float', 'Descrição': 'Volume programado para produção no mês — horizonte de 1 mês (KHL)'},
    {'Nome da Coluna': 'Nominal [grf/hora]',    'Tipo': 'float', 'Descrição': 'Capacidade nominal da linha produtiva em gramas por hora'},
    {'Nome da Coluna': 'Capacity Semanal [hl]', 'Tipo': 'float', 'Descrição': 'Capacidade disponível da linha produtiva em hectolitros por semana'},
    {'Nome da Coluna': 'Crítica PCPs',          'Tipo': 'str',   'Descrição': 'Restrições críticas apontadas pelo time de Planejamento e Controle da Produção'},
])

display(producao)

### 3.3 Estoque

In [ ]:
estoque = pd.DataFrame([
    {'Nome da Coluna': 'EI Mês',     'Tipo': 'float', 'Descrição': 'Estoque Inicial do mês em hectolitros (KHL)'},
    {'Nome da Coluna': 'EI Semana',  'Tipo': 'float', 'Descrição': 'Estoque Inicial da semana em hectolitros (KHL)'},
    {'Nome da Coluna': 'EF Semana',  'Tipo': 'float', 'Descrição': 'Estoque Final da semana. Fórmula: EI + Transferências + Trânsito − Demanda (KHL)'},
    {'Nome da Coluna': 'EFM',        'Tipo': 'float', 'Descrição': 'Estoque Final do mês em hectolitros (KHL)'},
    {'Nome da Coluna': 'Suf.ini(d)', 'Tipo': 'int',   'Descrição': 'Suficiência inicial em dias: EI / linear de demanda diária'},
    {'Nome da Coluna': 'Suf. f (d)', 'Tipo': 'int',   'Descrição': 'Suficiência final em dias: EF / linear de demanda diária. DOI mínimo exigido: 12 dias'},
    {'Nome da Coluna': 'Trânsito',   'Tipo': 'float', 'Descrição': 'Volume em trânsito via cabotagem ainda não contabilizado no estoque local (KHL)'},
])

display(estoque)

### 3.4 Transferências

In [ ]:
transferencias = pd.DataFrame([
    {'Nome da Coluna': 'Transf. Malha',      'Tipo': 'float', 'Descrição': 'Volume líquido de transferência na malha. Positivo (+): recebe; Negativo (−): exporta (KHL)'},
    {'Nome da Coluna': 'Transf. Interna',    'Tipo': 'float', 'Descrição': 'Volume recebido (+) ou enviado (−) dentro da própria região NENO (KHL)'},
    {'Nome da Coluna': 'Transf. Ext (Cabo)', 'Tipo': 'float', 'Descrição': 'Volume recebido (+) ou enviado (−) para fora da região via cabotagem. Lead time: 25 dias (KHL)'},
    {'Nome da Coluna': 'Transf. Ext (Rodo)', 'Tipo': 'float', 'Descrição': 'Volume recebido (+) ou enviado (−) para fora da região via rodoviário. Lead time: 6 dias, custo 60% maior que cabo (KHL)'},
    {'Nome da Coluna': 'Modal',              'Tipo': 'str',   'Descrição': 'Modal de transporte da transferência (Cabotagem ou Rodoviário)'},
    {'Nome da Coluna': 'Regional Origem',    'Tipo': 'str',   'Descrição': 'Região de origem do produto (ex: REG SE, REG NE)'},
    {'Nome da Coluna': 'Origem',             'Tipo': 'str',   'Descrição': 'Unidade produtiva de origem (ex: BR16 - F. Jacareí - SP)'},
    {'Nome da Coluna': 'Geo Destino',        'Tipo': 'str',   'Descrição': 'Região geográfica de destino da transferência'},
    {'Nome da Coluna': 'Desc Destino',       'Tipo': 'str',   'Descrição': 'Descrição do ponto de destino (ex: CDR Bahia, F. Fonte Mata)'},
])

display(transferencias)

### 3.5 Demanda

In [ ]:
demanda = pd.DataFrame([
    {'Nome da Coluna': 'Demanda', 'Tipo': 'float', 'Descrição': 'Volume previsto de venda para o período em hectolitros (KHL)'},
])

display(demanda)

### 3.6 Custos

In [ ]:
custos = pd.DataFrame([
    {'Nome da Coluna': 'Reais / HL',              'Tipo': 'float', 'Descrição': 'Custo unitário de transferência em reais por hectolitro (R$/HL)'},
    {'Nome da Coluna': 'Custo de Transferências', 'Tipo': 'float', 'Descrição': 'Custo total das transferências externas realizadas no período (R$)'},
    {'Nome da Coluna': 'MACOs - Produção Interna','Tipo': 'float', 'Descrição': 'Margem de contribuição quando a produção ocorre dentro da região NENO (R$)'},
    {'Nome da Coluna': 'Custo de Produção',       'Tipo': 'float', 'Descrição': 'Custo total de produção do SKU no período (R$)'},
])

display(custos)

### 3.7 Semanas de Planejamento

In [ ]:
semanas = pd.DataFrame([
    {'Nome da Coluna': 'W0 - 02/02/2026', 'Tipo': 'date', 'Descrição': 'Semana 0 — semana corrente de referência do planejamento'},
    {'Nome da Coluna': 'W1 - 09/02/2026', 'Tipo': 'date', 'Descrição': 'Semana 1 — primeira semana de planejamento futuro'},
    {'Nome da Coluna': 'W2 - 16/02/2026', 'Tipo': 'date', 'Descrição': 'Semana 2 — segunda semana de planejamento futuro'},
    {'Nome da Coluna': 'W3 - 23/02/2026', 'Tipo': 'date', 'Descrição': 'Semana 3 — terceira semana de planejamento futuro'},
])

display(semanas)

### 3.8 Unidades Produtivas e CDRs

In [ ]:
unidades = pd.DataFrame([
    {'Código':  'AQ541 / BR03', 'Nome': 'F. Aquiraz',              'Tipo': 'Cervejaria',              'Localização': 'NO Centro (CE)', 'Atende':                    'Todo o Nordeste e Manaus'},
    {'Código':  'NS541 / BR31', 'Nome': 'F. Pernambuco',           'Tipo': 'Cervejaria',              'Localização': 'NE Norte (PE)',  'Atende':                    'Todo o Nordeste'},
    {'Código':  'BR06',         'Nome': 'CDR João Pessoa',         'Tipo': 'Centro de Distribuição',  'Localização': 'NE Norte (PB)',  'Atende':                    'NE Norte, NO Centro, Mapapi'},
    {'Código':  'BR04',         'Nome': 'CDR Camaçari / Bahia',    'Tipo': 'Centro de Distribuição',  'Localização': 'NE Sul (BA)',    'Atende':                    'NE Sul'},
    {'Código':  'BR16',         'Nome': 'F. Jacareí',              'Tipo': 'Cervejaria',              'Localização': 'SP',             'Atende':                    'Todo o Brasil (SPLNs)'},
    {'Código':  'BR23',         'Nome': 'F. Jaguariúna',           'Tipo': 'Cervejaria',              'Localização': 'SP',             'Atende':                    'Todo o Brasil (SPLNs)'},
    {'Código':  'UB541',        'Nome': 'F. Uberlândia',           'Tipo': 'Cervejaria',              'Localização': 'MG',             'Atende':                    'MG e NO Araguaia'},
])

display(unidades)

### 3.9 SKUs da Linha Long Neck Premium (NENO)

In [ ]:
skus = pd.DataFrame([
    {'SKU': 'PATAGONIA AMBER LAGER LN355ML CX12',          'Linha Produtiva': 'AQ541',        'Restrição': 'Sem restrição crítica identificada'},
    {'SKU': 'GOOSE ISLAND MIDWAY NAC LN 355ML CX C/12',    'Linha Produtiva': 'NS541',        'Restrição': '⚠️ NO TETO da capacidade de líquido — NÃO pode aumentar produção'},
    {'SKU': 'MALZBIER BRAHMA LN355ML SIX PAC BAND C/4',    'Linha Produtiva': 'AQ541/NS541',  'Restrição': 'Foco do aumento de demanda (+30% em Fev, +10% TT LN em Mar)'},
    {'SKU': 'COLORADO LAGER LN 355ML CX C/12',             'Linha Produtiva': 'AQ541/NS541',  'Restrição': 'Sem restrição crítica identificada'},
    {'SKU': 'BRAHMA CHOPP ZERO LN 355 SIXP CX CART C4',    'Linha Produtiva': 'NS541',        'Restrição': 'Sem restrição crítica identificada'},
    {'SKU': 'SKOL BEATS SENSES LN 269ML SIXPACK SH C4',    'Linha Produtiva': 'NS541',        'Restrição': 'Sem restrição crítica identificada'},
    {'SKU': 'BUDWEISER ZERO LN 330ML SIX-PACK SH C/4',     'Linha Produtiva': 'NS541',        'Restrição': 'Sem restrição crítica identificada'},
])

display(skus)

---

## 4. Notas Operacionais Importantes

| Parâmetro | Valor |
|---|---|
| DOI mínimo NENO | **12 dias** |
| Lead time Cabotagem | **25 dias** |
| Lead time Rodoviário | **6 dias** |
| Custo Rodoviário vs Cabo | **+60%** |
| Risco avaria Rodoviário | **+5%** |
| Capacidade AQ541 | **50 khl/mês** |
| Capacidade NS541 | **108 khl/mês** |
| Nova demanda Fev NENO | **192 KHL** |
| Nova demanda Mar NENO | **211 KHL** |
| BIAS médio GEOs (últimos 3 meses) | **+9%** |

---
*Próxima etapa: Análise Univariada*